# Petrol (PMS) Price Extraction

Extracts monthly petrol price data from NBS "Tables" files and builds a single wide-format master table (State × Month).

In [16]:
import pandas as pd 

## Extraction Function

Each raw file contains 3 price columns per state: same month last year, previous month, and current month. This function extracts all three, labels them with real dates, separates the 37 states from the national summary rows, and standardizes the "Nasarawa" spelling (NBS used "Nassarawa" in some files).


In [17]:
def load_pms_file(file_path):
    xls = pd.ExcelFile(file_path)
    sheet = xls.sheet_names[0]
    df = pd.read_excel(file_path, sheet_name=sheet)
    cleaned_df = df.iloc[:, 0:4].copy()

    new_cols = ["State"]

    for col in cleaned_df.columns[1:]:
        if isinstance(col, (pd.Timestamp, pd.DatetimeIndex)):
            new_cols.append(col.strftime("%Y-%m"))
        elif hasattr(col, "strftime"):
            new_cols.append(col.strftime("%Y-%m"))
        else:
            new_cols.append(str(col))

    cleaned_df.columns = new_cols

    states_df = cleaned_df.iloc[0:37].copy()
    states_df["State"] = states_df["State"].replace("Nassarawa", "Nasarawa")
    summary_df = cleaned_df.iloc[37:].copy()

    return {
        "states": states_df,
        "summary": summary_df,
        "months": list(cleaned_df.columns[1:]),
    }

## Merging Multiple Months

Since each file contributes 3 date columns (not just 1), some months overlap across different files' year-ago/previous-month columns. This function merges all files together on State, skipping any month column that's already been added, so overlapping months aren't duplicated.

In [18]:
def merge_to_master(months, key="states"):
    master_df = None

    for month_name, month_data in months.items():
        df = month_data[key]
        piece = df.iloc[:, 0:4].copy()
        piece.columns = ["State"] + month_data["months"]

        if master_df is None:
            master_df = piece
        else:
            existing_cols = set(master_df.columns) - {"State"}
            new_piece_cols = [
                c for c in piece.columns if c not in existing_cols
            ]
            piece = piece[new_piece_cols]
            master_df = master_df.merge(piece, on="State", how="outer")

        date_cols = sorted([c for c in master_df.columns if c != "State"])
        master_df = master_df[["State"] + date_cols]

    return master_df

## Loading All Files

Loading every monthly file from November 2024 to May 2026. Because each file also contains a year-ago column, this range effectively extends the usable data back to November 2023.

In [19]:
months = {
    "2026-05" : load_pms_file("../data/raw/petrol/Fuel_Report_May_2026.xlsx"),
    "2026-04" : load_pms_file("../data/raw/petrol/Fuel_Report_April_2026.xlsx"),
    "2026-03" : load_pms_file("../data/raw/petrol/Fuel_Report_March_2026.xlsx"),
    "2026-02" : load_pms_file("../data/raw/petrol/Fuel_Report_February_2026.xlsx"),
    "2026-01" : load_pms_file("../data/raw/petrol/Fuel_Report_January_2026.xlsx"),
    "2025-12" : load_pms_file("../data/raw/petrol/Fuel_Report_December_2025.xlsx"),
    "2025-11" : load_pms_file("../data/raw/petrol/Fuel_Report_November_2025.xlsx"),
    "2025-10" : load_pms_file("../data/raw/petrol/Fuel_Report_October_2025.xlsx"),
    "2025-09" : load_pms_file("../data/raw/petrol/Fuel_Report_September_2025.xlsx"),
    "2025-08" : load_pms_file("../data/raw/petrol/Fuel_Report_August_2025.xlsx"),
    "2025-07" : load_pms_file("../data/raw/petrol/Fuel_Report_July_2025.xlsx"),
    "2025-06" : load_pms_file("../data/raw/petrol/Fuel_Report_June_2025.xlsx"),
    "2025-05" : load_pms_file("../data/raw/petrol/Fuel_Report_May_2025.xlsx"),
    "2025-04" : load_pms_file("../data/raw/petrol/Fuel_Report_April_2025.xlsx"),
    "2025-03" : load_pms_file("../data/raw/petrol/Fuel_Report_March_2025.xlsx"),
    "2025-02" : load_pms_file("../data/raw/petrol/Fuel_Report_February_2025.xlsx"),
    "2025-01" : load_pms_file("../data/raw/petrol/Fuel_Report_January_2025.xlsx"),
    "2024-12" : load_pms_file("../data/raw/petrol/Fuel_Report_December_2024.xlsx"),
    "2024-11" : load_pms_file("../data/raw/petrol/Fuel_Report_November_2024.xlsx"),
    
}

master_states_df = merge_to_master(months, key="states")
master_summary_df = merge_to_master(months, key="summary")


## Save Master Table

Saving the merged state-level petrol dataset to petrol_master.csv.

In [20]:
master_states_df.to_csv("../data/processed/petrol_master.csv", index=False)


In [26]:
master_states_df


,State,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,...,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05
0,Abia,654.357143,690.526316,687.500000,688.620000,710.833333,706.000000,770.000000,794.909091,797.420586,...,975.000000,1000.250000,1012.500000,1040.422500,1032.118113,1054.250000,1095.194146,1247.838321,1557.392740,1653.910020
1,Abuja,640.000000,663.333333,632.124444,653.200000,695.600000,670.000000,662.731111,696.000000,677.000000,...,1009.916667,962.688333,1039.166667,1065.743333,1033.779567,959.611292,1006.329848,1268.490545,1553.745941,1602.640094
2,Adamawa,638.375000,745.714286,671.400000,680.420000,735.000000,741.666667,798.600000,741.666667,809.733337,...,1025.258333,995.100917,1053.010000,1055.718333,1052.749942,1090.194792,1096.632464,1350.732165,1417.583752,1469.825139
3,Akwa Ibom,675.000000,671.666667,677.000000,678.000000,704.444444,692.500000,790.000000,807.777778,673.750000,...,973.506000,957.140000,1064.325000,1064.325000,1059.426625,1112.782254,1109.436341,1301.516137,1585.597407,1629.080974
4,Anambra,659.166667,680.000000,680.000000,685.170000,701.250000,707.857143,766.944444,779.419643,781.994542,...,1055.267337,989.021015,1068.238125,1068.238125,1066.596934,980.793750,1077.507992,1441.224280,1515.722025,1519.717866
5,Bauchi,650.000000,675.285714,650.000000,688.571429,707.142857,718.750000,693.805556,718.750000,816.250000,...,977.352222,956.292222,1078.344444,1082.603333,1079.612772,967.006106,1005.451759,1290.472879,1589.067064,1715.472457
6,Bayelsa,656.666667,667.111111,673.000000,680.000000,702.000000,689.000000,791.000000,797.272727,684.000000,...,973.095000,929.055250,1073.912500,1073.912500,1058.078812,968.887500,1050.027500,1252.484796,1536.810187,1591.376812
7,Benue,668.750000,658.888889,632.844577,652.727273,700.700000,713.636364,882.222222,854.545455,846.945821,...,1031.530000,1096.968500,1071.378850,1080.918850,1034.427979,1059.490000,1069.397136,1303.679429,1582.754184,1698.567931
8,Borno,625.750000,622.714286,657.272727,680.220000,721.750000,743.909091,800.318182,743.909091,829.464286,...,1145.500000,1045.000000,1101.625000,1133.861250,1072.042187,1079.500000,1108.085024,1375.162292,1476.115339,1551.069142
9,Cross River,666.000000,679.000000,663.333333,665.625000,710.000000,682.777778,740.000000,793.333333,677.777778,...,998.528571,964.107143,1031.250000,1047.436250,1051.502294,1171.765185,1068.606344,1365.022638,1514.141850,1551.214680
